In [1]:
#!/usr/bin/env python
# coding: utf-8

# Library Import
import pandas as pd
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Importing Dataset
file_path = "../data/medical_clean.csv"
df = pd.read_csv(file_path)

# Display df
print(df.shape)
print(df.info())
print(df.head())

(10000, 50)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 50 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CaseOrder           10000 non-null  int64  
 1   Customer_id         10000 non-null  object 
 2   Interaction         10000 non-null  object 
 3   UID                 10000 non-null  object 
 4   City                10000 non-null  object 
 5   State               10000 non-null  object 
 6   County              10000 non-null  object 
 7   Zip                 10000 non-null  int64  
 8   Lat                 10000 non-null  float64
 9   Lng                 10000 non-null  float64
 10  Population          10000 non-null  int64  
 11  Area                10000 non-null  object 
 12  TimeZone            10000 non-null  object 
 13  Job                 10000 non-null  object 
 14  Children            10000 non-null  int64  
 15  Age                 10000 non-null  int64 

In [2]:
# Columns to keep
keep_cols = [
    # Medical conditions/history
    
    "HighBlood", "Stroke", "Diabetes", "Overweight", "Arthritis", 
    "Hyperlipidemia", "BackPain", "Anxiety", "Allergic_rhinitis", 
    "Reflux_esophagitis", "Asthma",
    
    # Demographic variables
    "Age", "Income", "Children", "Gender", "Marital", "Area",
    
    # Additional relevant variables
    "Complication_risk", "Initial_admin", "Services", "Initial_days", "Doc_visits",
    
    # Target variable
    "ReAdmis"
]

# Keep only the selected columns
df = df[keep_cols]

# Quick check
print(df.info())
print(f"Dataset now has {df.shape[0]} rows and {df.shape[1]} columns")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   HighBlood           10000 non-null  object 
 1   Stroke              10000 non-null  object 
 2   Diabetes            10000 non-null  object 
 3   Overweight          10000 non-null  object 
 4   Arthritis           10000 non-null  object 
 5   Hyperlipidemia      10000 non-null  object 
 6   BackPain            10000 non-null  object 
 7   Anxiety             10000 non-null  object 
 8   Allergic_rhinitis   10000 non-null  object 
 9   Reflux_esophagitis  10000 non-null  object 
 10  Asthma              10000 non-null  object 
 11  Age                 10000 non-null  int64  
 12  Income              10000 non-null  float64
 13  Children            10000 non-null  int64  
 14  Gender              10000 non-null  object 
 15  Marital             10000 non-null  object 
 16  Area 

In [3]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Handle missing values if any exist
df = df.dropna() 
print(f"Dataset shape after handling missing values: {df.shape}")

Missing values per column:
HighBlood             0
Stroke                0
Diabetes              0
Overweight            0
Arthritis             0
Hyperlipidemia        0
BackPain              0
Anxiety               0
Allergic_rhinitis     0
Reflux_esophagitis    0
Asthma                0
Age                   0
Income                0
Children              0
Gender                0
Marital               0
Area                  0
Complication_risk     0
Initial_admin         0
Services              0
Initial_days          0
Doc_visits            0
ReAdmis               0
dtype: int64
Dataset shape after handling missing values: (10000, 23)


In [4]:
for column in df.columns:
    print(f"Unique values in column '{column}':")
    print(df[column].unique())
    print()

Unique values in column 'HighBlood':
['Yes' 'No']

Unique values in column 'Stroke':
['No' 'Yes']

Unique values in column 'Diabetes':
['Yes' 'No']

Unique values in column 'Overweight':
['No' 'Yes']

Unique values in column 'Arthritis':
['Yes' 'No']

Unique values in column 'Hyperlipidemia':
['No' 'Yes']

Unique values in column 'BackPain':
['Yes' 'No']

Unique values in column 'Anxiety':
['Yes' 'No']

Unique values in column 'Allergic_rhinitis':
['Yes' 'No']

Unique values in column 'Reflux_esophagitis':
['No' 'Yes']

Unique values in column 'Asthma':
['Yes' 'No']

Unique values in column 'Age':
[53 51 78 22 76 50 40 48 55 64 41 45 85 57 44 54 72 84 86 52 31 37 19 75
 70 69 56 32 27 65 25 79 24 89 58 60 33 83 66 73 35 43 63 36 39 29 20 34
 47 18 59 82 26 74 38 28 77 30 87 23 80 71 68 67 88 62 49 21 61 81 42 46]

Unique values in column 'Income':
[86575.93 46805.99 14370.14 ... 65917.81 29702.32 62682.63]

Unique values in column 'Children':
[ 1  3  0  7  2  4 10  5  8  6  9]

Unique 

In [5]:
# Start from your selected columns
df_encoded = df[keep_cols].copy()

# All binary yes/no columns (medical + target)
binary_cols = [
    
    'HighBlood','Stroke','Diabetes','Overweight','Arthritis','Hyperlipidemia',
    'BackPain','Anxiety','Allergic_rhinitis','Reflux_esophagitis','Asthma','ReAdmis'
]

# Normalize and map yes/no -> 1/0
for col in binary_cols:
    df_encoded[col] = (
        df_encoded[col]
        .astype(str).str.strip().str.lower()
        .map({'yes': 1, 'no': 0})
    )

# One-hot encode multi-category variables
df_encoded = pd.get_dummies(
    df_encoded,
    columns=['Gender','Marital','Area','Complication_risk','Initial_admin','Services'],
    drop_first=False
)

# Convert all boolean columns to integers (0/1)
bool_cols = df_encoded.select_dtypes(include=['bool']).columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)



# Sanity checks
print(df_encoded[binary_cols].isna().sum())   # we want all to be 0
for col in binary_cols:
    print(col, df_encoded[col].value_counts(dropna=False).to_dict())


HighBlood             0
Stroke                0
Diabetes              0
Overweight            0
Arthritis             0
Hyperlipidemia        0
BackPain              0
Anxiety               0
Allergic_rhinitis     0
Reflux_esophagitis    0
Asthma                0
ReAdmis               0
dtype: int64
HighBlood {0: 5910, 1: 4090}
Stroke {0: 8007, 1: 1993}
Diabetes {0: 7262, 1: 2738}
Overweight {1: 7094, 0: 2906}
Arthritis {0: 6426, 1: 3574}
Hyperlipidemia {0: 6628, 1: 3372}
BackPain {0: 5886, 1: 4114}
Anxiety {0: 6785, 1: 3215}
Allergic_rhinitis {0: 6059, 1: 3941}
Reflux_esophagitis {0: 5865, 1: 4135}
Asthma {0: 7107, 1: 2893}
ReAdmis {0: 6331, 1: 3669}


In [6]:
# Save the cleaned dataset
df_encoded.to_csv('medical_cleaned_encoded.csv', index=False)
print("Cleaned dataset saved as 'medical_cleaned_encoded.csv'")
print(f"Dataset shape: {df_encoded.shape}")

Cleaned dataset saved as 'medical_cleaned_encoded.csv'
Dataset shape: (10000, 38)


In [7]:
# Separate features (X) and target (y)
X = df_encoded.drop('ReAdmis', axis=1)
y = df_encoded['ReAdmis']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Features shape: (10000, 37)
Target distribution:
ReAdmis
0    6331
1    3669
Name: count, dtype: int64


In [8]:
# Split into train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Split temp into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

Train: (7000, 37) Validation: (1500, 37) Test: (1500, 37)


In [9]:
# Save features and labels for each split
X_train.to_csv("X_train.csv", index=False)
y_train.to_csv("y_train.csv", index=False)

X_val.to_csv("X_val.csv", index=False)
y_val.to_csv("y_val.csv", index=False)

X_test.to_csv("X_test.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Datasets saved as CSV files:")
print("X_train.csv, y_train.csv, X_val.csv, y_val.csv, X_test.csv, y_test.csv")

Datasets saved as CSV files:
X_train.csv, y_train.csv, X_val.csv, y_val.csv, X_test.csv, y_test.csv


In [10]:
# Train model
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

# Predict on validation set
y_val_pred = rf.predict(X_val)
y_val_prob = rf.predict_proba(X_val)[:, 1]

# Metrics
print("Accuracy :", round(accuracy_score(y_val, y_val_pred), 3))
print("Precision:", round(precision_score(y_val, y_val_pred), 3))
print("Recall   :", round(recall_score(y_val, y_val_pred), 3))
print("F1-score :", round(f1_score(y_val, y_val_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_prob), 3))

print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred, digits=3))

Accuracy : 0.981
Precision: 0.973
Recall   : 0.976
F1-score : 0.975
ROC-AUC  : 0.998

Confusion Matrix:
 [[934  15]
 [ 13 538]]

Classification Report:
               precision    recall  f1-score   support

           0      0.986     0.984     0.985       949
           1      0.973     0.976     0.975       551

    accuracy                          0.981      1500
   macro avg      0.980     0.980     0.980      1500
weighted avg      0.981     0.981     0.981      1500



In [11]:
from sklearn.model_selection import GridSearchCV

# Define base model
rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [200, 500, 800],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# GridSearch with 5-fold CV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score (ROC-AUC):", grid_search.best_score_)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


Best Parameters: {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 800}
Best CV Score (ROC-AUC): 0.9982006225152714


In [12]:
best_rf = RandomForestClassifier(
    n_estimators=800,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

# Predict on test set
y_test_pred = best_rf.predict(X_test)
y_test_prob = best_rf.predict_proba(X_test)[:, 1]

# Metrics
print("Accuracy :", round(accuracy_score(y_test, y_test_pred), 3))
print("Precision:", round(precision_score(y_test, y_test_pred), 3))
print("Recall   :", round(recall_score(y_test, y_test_pred), 3))
print("F1-score :", round(f1_score(y_test, y_test_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_test_prob), 3))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred, digits=3))

Accuracy : 0.98
Precision: 0.964
Recall   : 0.982
F1-score : 0.973
ROC-AUC  : 0.999

Confusion Matrix:
 [[930  20]
 [ 10 540]]

Classification Report:
               precision    recall  f1-score   support

           0      0.989     0.979     0.984       950
           1      0.964     0.982     0.973       550

    accuracy                          0.980      1500
   macro avg      0.977     0.980     0.979      1500
weighted avg      0.980     0.980     0.980      1500



## Feature Timing Sensitivity Check

The full model includes `Initial_days`, which is known after the hospital stay has already unfolded. That can be valid for discharge-time risk stratification, but it should not be interpreted as a clean pre-admission readmission predictor. The check below quantifies how much the headline performance depends on that timing-sensitive feature.


In [13]:
# Feature timing sensitivity check
feature_importance = (
    pd.Series(best_rf.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

print("Top 10 feature importances from the tuned full model:")
display(feature_importance.head(10).to_frame("importance"))

# Re-run the tuned model without Initial_days, which is a post-admission / discharge-time feature.
X_timing_check = df_encoded.drop(columns=["ReAdmis", "Initial_days"])
y_timing_check = df_encoded["ReAdmis"]

X_train_t, X_temp_t, y_train_t, y_temp_t = train_test_split(
    X_timing_check, y_timing_check, test_size=0.3, random_state=42, stratify=y_timing_check
)
X_val_t, X_test_t, y_val_t, y_test_t = train_test_split(
    X_temp_t, y_temp_t, test_size=0.5, random_state=42, stratify=y_temp_t
)

timing_rf = RandomForestClassifier(
    n_estimators=800,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="log2",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
timing_rf.fit(X_train_t, y_train_t)

y_timing_pred = timing_rf.predict(X_test_t)
y_timing_prob = timing_rf.predict_proba(X_test_t)[:, 1]

print("\nTuned model without Initial_days:")
print("Accuracy :", round(accuracy_score(y_test_t, y_timing_pred), 3))
print("Precision:", round(precision_score(y_test_t, y_timing_pred), 3))
print("Recall   :", round(recall_score(y_test_t, y_timing_pred), 3))
print("F1-score :", round(f1_score(y_test_t, y_timing_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test_t, y_timing_prob), 3))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_t, y_timing_pred))

print(
    "\nInterpretation: the full model is best described as a discharge-time risk "
    "stratification model. Its very high ROC-AUC is driven primarily by Initial_days, "
    "so it should not be presented as a pre-admission prediction model."
)


Top 10 feature importances from the tuned full model:


,importance
Initial_days,0.781106
Income,0.031201
Age,0.027453
Children,0.015855
Doc_visits,0.013329
Arthritis,0.005338
Allergic_rhinitis,0.005313
BackPain,0.005297
Anxiety,0.005287
HighBlood,0.005264



Tuned model without Initial_days:
Accuracy : 0.631
Precision: 0.429
Recall   : 0.016
F1-score : 0.032
ROC-AUC  : 0.498

Confusion Matrix:
 [[938  12]
 [541   9]]

Interpretation: the full model is best described as a discharge-time risk stratification model. Its very high ROC-AUC is driven primarily by Initial_days, so it should not be presented as a pre-admission prediction model.
